In [ ]:
from google.colab import files
uploaded = files.upload()   # select all 4 CSVs
print(list(uploaded.keys()))

In [ ]:
import os
print([f for f in os.listdir('.') if f.endswith('.csv')])

In [ ]:
import pandas as pd

F_CODING = 'confirmatory_coding_final.csv'
F_MATH   = 'math_with_hinglish.csv'
F_FACT   = 'reasearch_paper_dataset.csv'
F_OPEN   = 'ARC_Open_Ended_Hinglish_38 (1).csv'

def read(f):
    try:
        return pd.read_csv(f, encoding='utf-8')
    except UnicodeDecodeError:           # the factual file is not UTF-8
        return pd.read_csv(f, encoding='cp1252')

cols = ['pair_id', 'task_type', 'difficulty', 'source', 'source_id', 'written_by', 'checked_by',
        'check_status', 'check_note', 'english', 'hinglish', 'hinglish_translated', 'hinglish_devanagari']

cod = read(F_CODING)   # already in the final format, IDs 101-138

f = read(F_FACT).dropna(subset=['id'])          # drops the blank row in the middle
fact = pd.DataFrame({
    'task_type': 'Factual',
    'difficulty': f['difficulty'].values,
    'source': f['difficulty'].map({'Easy': 'ARC-Easy', 'Hard': 'ARC-Challenge'}).values,
    'source_id': f['id'].values,
    'english': f['english'].values,
    'hinglish': f['hinglish'].values})

mth = read(F_MATH)[['task_type', 'difficulty', 'source', 'source_id', 'english', 'hinglish']]

o = read(F_OPEN)
opn = pd.DataFrame({
    'task_type': 'Open-Ended',
    'difficulty': o['Source difficulty'].values,
    'source': 'Written',
    'source_id': '',
    'english': o['Open-ended question (English)'].values,
    'hinglish': o['Hinglish translation'].values})

rest = pd.concat([fact, mth, opn], ignore_index=True)
for c in ['written_by', 'checked_by', 'check_note', 'hinglish_translated', 'hinglish_devanagari']:
    rest[c] = ''
rest['check_status'] = 'PENDING'
start = int(cod['pair_id'].max()) + 1
rest['pair_id'] = range(start, start + len(rest))

master = pd.concat([cod[cols], rest[cols]], ignore_index=True).fillna('')
print(master.groupby(['task_type', 'difficulty']).size())

In [ ]:
from datasets import load_dataset
import re

def norm(s):
    s = str(s).replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    return re.sub(r'\s+', ' ', s).strip().lower()

lookup = {}
for cfg in ['ARC-Easy', 'ARC-Challenge']:
    d = load_dataset('allenai/ai2_arc', cfg, split='test').to_pandas()
    for i, q in zip(d['id'], d['question']):
        lookup[norm(q)] = (cfg, i)

n = 0
for idx in master.index[master['task_type'] == 'Factual']:
    h = lookup.get(norm(master.at[idx, 'english']))
    if h:
        master.at[idx, 'source'], master.at[idx, 'source_id'] = h
        n += 1
    else:
        print('no match:', master.at[idx, 'pair_id'])
print('matched', n, 'of 38')

In [ ]:
from datasets import load_dataset
import re

def norm(s):
    s = str(s).replace('\u2019', "'").replace('\u2018', "'").replace('\u201c', '"').replace('\u201d', '"')
    return re.sub(r'\s+', ' ', s).strip().lower()

lookup = {}   # question text -> (config, id, split)
for cfg in ['ARC-Easy', 'ARC-Challenge']:
    for sp in ['train', 'validation', 'test']:
        d = load_dataset('allenai/ai2_arc', cfg, split=sp).to_pandas()
        for i, q in zip(d['id'], d['question']):
            lookup[norm(q)] = (cfg, i, sp)

found = {}
for idx in master.index[master['task_type'] == 'Factual']:
    h = lookup.get(norm(master.at[idx, 'english']))
    if h:
        found[idx] = h
        master.at[idx, 'source'], master.at[idx, 'source_id'] = h[0], h[1]
    else:
        print('no match:', master.at[idx, 'pair_id'])

print('matched', len(found), 'of 38')
print(pd.Series([h[2] for h in found.values()]).value_counts())   # which split
# does Easy/Hard agree with ARC-Easy/ARC-Challenge?
bad = [master.at[i, 'pair_id'] for i in found
       if {'Easy': 'ARC-Easy', 'Hard': 'ARC-Challenge'}[master.at[i, 'difficulty']] != found[i][0]]
print('difficulty label disagrees with ARC set for pair_ids:', bad)

In [ ]:
for q, (cfg, i, sp) in lookup.items():
    if 'digestion process begins' in q:
        print(cfg, i, sp, '|', q)

In [ ]:
master.loc[master['pair_id'] == 163, ['source', 'source_id']] = ['ARC-Easy', 'Mercury_SC_400329']
master.loc[master['pair_id'] == 163, 'check_note'] = 'English reworded from ARC stem'
print(master.loc[master['pair_id'] == 163, ['source', 'source_id', 'english']].to_string())

In [ ]:
import re

# collapse double spaces in English
master['english'] = master['english'].str.replace(r' {2,}', ' ', regex=True)

# pair 197: LaTeX-style quotes
for col in ['english', 'hinglish']:
    master.loc[master['pair_id'] == 197, col] = (
        master.loc[master['pair_id'] == 197, col].str.replace("`SIXTEEN'", "'SIXTEEN'", regex=False))

# pair 145, 149, 163 Hinglish typos
master.loc[master['pair_id'] == 145, 'hinglish'] = (
    "Rain forests mein tree species kisi bhi aur biome se zyada hoti hain. "
    "Phir bhi scientists ne paaya hai ki forest floor ki mitti mein nutrients kaafi kam hote hain. "
    "Iska kya karan ho sakta hai?")
master.loc[master['pair_id'] == 149, 'hinglish'] = (
    master.loc[master['pair_id'] == 149, 'hinglish'].str.replace('mukable', 'muqable', regex=False))
master.loc[master['pair_id'] == 163, 'hinglish'] = (
    master.loc[master['pair_id'] == 163, 'hinglish'].str.replace('nsaanon', 'Insaanon', regex=False))

In [ ]:
print('mojibake:', master.astype(str).apply(lambda c: c.str.contains('Ã|Â|â€', regex=True)).sum().sum())
print('BOM check after save below')

In [ ]:
# who wrote the Hinglish (and the English for 'Written' rows), per task type
WRITER = {'Coding': 'A', 'Reasoning': 'R', 'Open-Ended': 'N', 'Factual': 'J'}
master['written_by'] = master['task_type'].map(WRITER)

# Coding rows were taken from public benchmarks, so name the benchmark in `source`
is_cod = master['task_type'] == 'Coding'
master.loc[is_cod & master['source_id'].str.startswith('MBPP-'), 'source'] = 'MBPP'
master.loc[is_cod & master['source_id'].str.startswith('LeetCode-'), 'source'] = 'LeetCode'

print(master['written_by'].value_counts())
print(master[is_cod].groupby('source').size())   # pair 130 (Classic-Knapsack) stays 'Written'

In [ ]:
print('rows:', len(master), '| duplicate ids:', master['pair_id'].duplicated().sum())
print('blank english:', (master['english'].str.strip() == '').sum(),
      '| blank hinglish:', (master['hinglish'].str.strip() == '').sum())
print('broken characters:', master.astype(str).apply(lambda c: c.str.contains('\ufffd')).sum().sum())

master.to_csv('confirmatoryv1_draft.csv', index=False, encoding='utf-8-sig')
print(open('confirmatoryv1_draft.csv', 'rb').read(3))
from google.colab import files
files.download('confirmatoryv1_draft.csv')